# 03 · Join Sofascore + Capology — France Ligue 1 23/24

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2023/24 de Ligue 1 francesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_france_2324.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_france_2324.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  525 jugadores | 116 columnas
Capology:   550 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   as monaco
   clermont foot
   olympique de marseille
   olympique lyonnais
   paris fc
   paris saint germain
   rc lens
   rc strasbourg
   rodez af
   stade brestois
   stade de reims
   stade rennais

En Capology pero no en Sofascore:
   brest
   clermont
   lens
   lyon
   marseille
   monaco
   psg
   reims
   rennes
   strasbourg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brest':'stade brestois',
            'clermont':'clermont foot',
            'lens':'rc lens',
            'lyon':'olympique lyonnais',
            'marseille':'olympique de marseille',
            'monaco':'as monaco',
            'psg':'paris saint germain',
            'reims':'stade de reims',
            'rennes':'stade rennais',
            'strasbourg':'rc strasbourg'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 441/525 (84.0%)
Sin emparejar: 84


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          8
Revisión media    (0.75 ≤ score < 0.90):   8
Revisión estricta (0.50 ≤ score < 0.75):   35
Revisión muy est. (score < 0.50):           31


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
13,Przemysław Frankowski,RC Lens,przemyslaw frankowski,0.976
1,Massamba Ndiaye,Clermont Foot,massamba n diaye,0.968
3,Radosław Majecki,AS Monaco,radoslaw majecki,0.968
54,Joseph N'Duquidi,Metz,joseph nduquidi,0.968
40,Emmanuel Emegha,RC Strasbourg,emanuel emegha,0.966
23,Darlin Yongwa,Lorient,darline yongwa,0.963
10,Marcin Bułka,Nice,marcin bulka,0.957
71,Luc Zogbé,Stade Brestois,luck zogbe,0.947


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
14,Mousa Tamari,Montpellier,mousa al tamari,0.889
43,Valentin Atangana Edoa,Stade de Reims,valentin atangana,0.872
48,François Régis Mughe,Olympique de Marseille,francois mughe,0.824
11,Abdoul Kone,Stade de Reims,amadou kone,0.818
34,Cheick Konaté,Clermont Foot,cheick oumar konate,0.812
42,Shavy Warren Babicka,Toulouse,shavy babicka,0.788
32,Enzo Tchato Mbiayi,Montpellier,enzo tchato,0.759
4,Wilfried Singo,AS Monaco,wilfried stephane singo,0.757


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['abdoul kone'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 7 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
15,Amir Murillo,Olympique de Marseille,michael murillo,0.741
52,Kévin Keben Biakolo,Toulouse,kevin keben,0.733
21,Ahmadou Bamba Dieng,Lorient,bamba dieng,0.733
7,Muhammed-Cham Saračević,Clermont Foot,muhammed cham,0.722
47,Ethan Mbappé,Paris Saint-Germain,kylian mbappe,0.720
61,Joel Mugisha Mvuka,Lorient,joel mvuka,0.714
18,Alexsandro Ribeiro,Lille,alexsandro,0.714
39,Cheick Tidiane Sabaly,Metz,cheikh sabaly,0.706
44,Gift Orban,Olympique Lyonnais,gift emmanuel orban,0.690
33,Dion Moise Sahi,RC Strasbourg,moise sahi dion,0.667


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['amir murillo',
                    'kevin keben biakolo',
                    'ahmadou bamba dieng',
                    'muhammed cham saracevic',
                    'joel mugisha mvuka',
                    'alexsandro ribeiro',
                    'cheick tidiane sabaly',
                    'gift orban',
                    'dion moise sahi',
                    'david pereira da costa',
                    'boubakar kouyate',
                    'andy logbo',
                    'angelo'
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 13


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
72,Chaim El Djebali,Olympique Lyonnais,mohamed el arouch,0.485
66,Ilhan Fakili,Clermont Foot,chrislain matsima,0.483
70,Sofiane Sidi Ali,Olympique de Marseille,iliman ndiaye,0.483
6,Othmane Maamma,Montpellier,mousa al tamari,0.483
69,Mohamed Bechikh,RC Strasbourg,maxime bastian,0.483
51,Enzo Mongo,Nantes,moses simon,0.476
58,Keyliane Abdallah,Olympique de Marseille,iliman ndiaye,0.467
65,Bassirou N'Diaye,Lorient,benjamin mendy,0.467
31,Farès Chaïbi,Toulouse,niklas schmidt,0.462
46,Mattéo Guendouzi,Olympique de Marseille,pape gueye,0.462


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 469/525 (89.3%)
Sin salario:     56


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 56


,player,team,minutesPlayed,appearances,goals,assists
0,Mamadou Coulibaly,AS Monaco,121,5,0,0
1,Saimon Bouabre,AS Monaco,45,1,0,0
2,Lucas Michal,AS Monaco,9,1,0,0
3,Mateusz Wieteska,Clermont Foot,175,2,1,0
4,Ilhan Fakili,Clermont Foot,99,3,0,0
5,Mohamed-Amine Bouchenna,Clermont Foot,27,2,0,0
6,Ivan M'Bahia,Clermont Foot,13,1,0,0
7,Abdellah Baallal,Clermont Foot,11,1,0,0
8,Steve Ngoura,Le Havre,247,12,0,1
9,Simon Ebonog,Le Havre,34,4,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  AS Monaco  —  SF sin salario:


,player,minutesPlayed
0,Lucas Michal,9
1,Mamadou Coulibaly,121
2,Saimon Bouabre,45


  CG plantilla completa:


,player,player_norm
0,Aleksandr Golovin,aleksandr golovin
1,Breel Embolo,breel embolo
2,Caio Henrique,caio henrique
3,Chrislain Matsima,chrislain matsima
4,Denis Zakaria,denis zakaria
5,Edan Diop,edan diop
6,Eliesse Ben Seghir,eliesse ben seghir
7,Eliot Matazo,eliot matazo
8,Folarin Balogun,folarin balogun
9,Gelson Martins,gelson martins



  Clermont Foot  —  SF sin salario:


...

,player,player_norm
0,Adama Bojang,adama bojang
1,Alexandre Olliero,alexandre olliero
2,Amadou Koné,amadou kone
3,Amine Salama,amine salama
4,Amir Richardson,amir richardson
5,Azor Matusiwa,azor matusiwa
6,Benjamin Stambouli,benjamin stambouli
7,Emmanuel Agbadou,emmanuel agbadou
8,Ibrahim Diakité,ibrahim diakite
9,Joseph Okumu,joseph okumu



  Toulouse  —  SF sin salario:


,player,minutesPlayed
0,Farès Chaïbi,56


  CG plantilla completa:


,player,player_norm
0,Álex Domínguez,alex dominguez
1,Aron Dønnum,aron dnnum
2,Bonota Traoré,bonota traore
3,César Gelabert,cesar gelabert
4,Christian Mawissa,christian mawissa
5,Cristian Cásseres Jr,cristian casseres jr
6,Denis Genreau,denis genreau
7,Frank Magri,frank magri
8,Gabriel Suazo,gabriel suazo
9,Guillaume Restes,guillaume restes


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 469/525 (89.3%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_france_2324.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_france_2324.csv
   Jugadores totales:  525
   Con salario:        469
   Sin salario (NaN):  56
   Columnas:           121
